# Seeded Plasma-Column Full Transport — Kr

Full-domain seeded neutralization run for **Kr** at **1e-6 Torr**.

> **Caution**: Analytic/data-driven seeded source estimate, not fully
> self-consistent proton-impact MCC. Label all results accordingly.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RUNS_DIR       = _ROOT / 'runs'
PLOTS_DIR      = _ROOT / 'plots'
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
SCRIPT = _ROOT / 'plasma_column_mcc_picmi_v7.py'
_DEFAULTS = {
    'script':               'plasma_column_mcc_picmi_v7.py',
    'case':                 'seeded_Kr',
    'gas':                  'Kr',
    'neutralization':       0.5,
    'mcc':                  'electron_impact',
    'pressure_torr':        '1e-6',
    'plasma_age [s]':       '2e-4',
    'max_steps':            120000,
    'diag_period':          5000,
    'reduced_diag_period':  100,
    'nx / ny / nz':         '32 / 32 / 256',
}
_OVERRIDES = {}  # e.g. {'max_steps': 500}
print_simulation_config(
    notebook_title='Seeded Full Transport — Kr',
    defaults=_DEFAULTS, overrides=_OVERRIDES,
    extra_info={'output root': str(RUNS_DIR / 'seeded_Kr')},
)


## 1. Configure


In [ ]:
MAX_STEPS      = 120000
DIAG_PERIOD    = 5000
REDUCED_PERIOD = 100
NX, NY, NZ     = 32, 32, 256
OUT_DIR = RUNS_DIR / 'seeded_Kr'
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, str(SCRIPT), '--run',
    '--output_dir',          str(OUT_DIR),
    '--gas',                 'Kr',
    '--neutralization',      '0.5',
    '--mcc',                 'electron_impact',
    '--pressure_torr',       '1e-6',
    '--plasma_age',          '2e-4',
    '--max_steps',           str(MAX_STEPS),
    '--diag_period',         str(DIAG_PERIOD),
    '--reduced_diag_period', str(REDUCED_PERIOD),
    '--reduced_diag_dir',    'reducedfiles/',
    '--nx', str(NX), '--ny', str(NY), '--nz', str(NZ),
    '--warpx_data_dir',      str(WARPX_DATA_DIR),
]
print('Command:', ' '.join(cmd))


## 2. Run


In [ ]:
# result = subprocess.run(cmd, check=True)
print('Ready — uncomment subprocess.run to launch.')


## 3. Load diagnostics


In [ ]:
from plasma_column.plotting import (
    setup_publication_style,
    plot_multi_case_neutralization,
    plot_neutralization_evolution,
    plot_particle_counts,
    plot_keff_over_k0,
    plot_species_growth_rates,
    plot_neutralization_panel,
    plot_bunched_beam_keff,
    plot_keff_pressure_scan,
    plot_radial_density_profile,
    plot_neutralization_vs_z,
    plot_phase_space,
    save_figure,
)
from plasma_column.diagnostics import (
    load_particle_number_diagnostic,
    compute_particle_number_metrics,
    DataLoader,
)
import warnings
setup_publication_style()
print('Plotting helpers loaded.')


In [ ]:
import warnings
_OUT = RUNS_DIR / 'seeded_Kr'
_diag_candidates = [
    _OUT / 'reducedfiles' / 'ParticleNumber_red.txt',
    _OUT / 'neutralization_from_particle_number.csv',
]
_diag = next((p for p in _diag_candidates if p.exists()), None)
if _diag:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        df_seeded = load_particle_number_diagnostic(_diag)
        df_seeded = compute_particle_number_metrics(df_seeded)
    print(f'Loaded {len(df_seeded)} steps from {_diag.name}')
    display(df_seeded.tail())
else:
    print('No diagnostic file yet — run the simulation first.')
    df_seeded = None


## 4. Diagnostic plots

> eta should rise from seed value. K_eff/K0 should decrease.


In [ ]:
if df_seeded is not None:
    plot_neutralization_panel(df_seeded, PLOTS_DIR, case_name='seeded_Kr')
    plot_species_growth_rates(df_seeded, PLOTS_DIR,
                              case_name='seeded_Kr', smooth_window=7)
    plot_bunched_beam_keff(
        df_seeded['time'].values * 1e9,
        df_seeded['eta_net'].values.clip(0, 1), PLOTS_DIR,
        case_name='seeded_Kr',
        bunching_factors=[1.0, 2.0, 3.0, 5.0],
    )
    plt.show()
    print('Final eta_net   =', df_seeded['eta_net'].iloc[-1])
    print('Final K_eff/K0  =', df_seeded['keff_over_k0'].iloc[-1])


## Physics checks (AGENTS.md)

- [ ] Beam velocity consistent with 30 keV proton energy
- [ ] Macroparticle weights are physically meaningful
- [ ] Species ordering in ParticleNumber is correct
- [ ] Ne and Ni increase for the correct physics reason
- [ ] K_eff/K0 never negative unless labelled overcompensation
- [ ] Beam envelope changes consistent with sign and magnitude of compensation
- [ ] Gas pressure and interaction length acceptable
- [ ] Results labelled as analytic/seeded (not self-consistent MCC)


In [ ]:
if df_seeded is not None:
    row = {
        'case':          'seeded_Kr',
        'gas':           'Kr',
        'pressure':      '1e-6 Torr',
        'n_steps':       len(df_seeded),
        'final_eta_e':   df_seeded['eta_electron_only'].iloc[-1],
        'final_eta_net': df_seeded['eta_net'].iloc[-1],
        'final_keff_K0': df_seeded['keff_over_k0'].iloc[-1],
    }
    display(pd.DataFrame([row]))
